# Step 16 — Validate network reconstruction and StationXML

Optional but recommended after step 15, especially after changing station evidence, epoch rules or response assumptions. This notebook replaces `network_history/test_reconstruction.py`. All test code is here; the external test runner loads these definitions without executing notebook run cells.

## Two different checks

1. **Controlled reconstruction tests** preserve the four former checks: StationXML schema and deliberately absent full responses; nominal sensor/attenuation calculations; epoch/undated low-gain rules; coordinate provenance. They use the current LEGACY reference evidence and a tiny synthetic header table, generating temporary test XML rather than overwriting production inventories. The 13-station/four-tilt-channel assertions are checks of the current reconstruction design, not universal network truths.
2. **Saved-product audit** reads the actual step-15 XML files under FAIR and reports schema validity, overlaps and response units. It does not scan waveforms or rebuild inventory products. Documentary shared endpoints are distinguished from positive-duration overlaps; the separate selection-safe copies should have no overlaps.

Passing tests establishes consistency with the encoded assumptions, not historical correctness. It cannot resolve missing ADC/telemetry calibration, sensor conflicts, station failure times, true MDETECT clock corrections or whether PI2L was recorded. Nominal response checks validate the voltage model numerically; they do not establish a calibrated digital-count response.

Test summary and artifact-audit reports are written under `FAIR/metadata/network_history/validation`. Source inputs, waveforms and existing XML files are unchanged. Temporary fixture files are removed automatically.

## 1. Configuration

Locate the pipeline and use its shared configuration. The next definition cell reads only the tagged function definitions from notebook 15: it does not execute its header scan, build or packaging cells.

In [ ]:
from pathlib import Path
import sys, json
from datetime import datetime, timezone
from IPython.display import display
candidates = [Path.cwd(), Path.cwd() / '10_conversion', *Path.cwd().parents]
PIPELINE = next((p for p in candidates if (p / 'pinatubo_steps').is_dir()), None)
if PIPELINE is None:
    raise FileNotFoundError('Start in the repository root or 10_conversion.')
if str(PIPELINE) not in sys.path:
    sys.path.insert(0, str(PIPELINE))
from pinatubo_steps.config import pipeline_paths
P = pipeline_paths()
NETWORK_OUTPUT = Path(P['NETWORK_HISTORY_OUTPUT'])
VALIDATION_OUTPUT = NETWORK_OUTPUT / 'validation'
RUN_SAVED_PRODUCT_AUDIT = True
print('Reference inputs:', P['NETWORK_HISTORY_INPUT'])
print('Saved network products:', NETWORK_OUTPUT)
print('Validation reports:', VALIDATION_OUTPUT)

## 2. Controlled test definitions

The test class creates a temporary one-channel header fixture and constructs both provisional and nominal-voltage inventories there. It tests a 6 dB attenuation change by evaluating response stages, not merely reading their descriptive sensitivity. It also preserves checks for unknown electronics, undated PI2L, tilt channel count, ordered epochs and explicitly sourced coordinates. No production processing runs in this definition cell.

In [ ]:
from pathlib import Path

import unittest

import numpy as np

from obspy import read_inventory,UTCDateTime

from obspy.io.stationxml.core import validate_stationxml

import json

if 'PIPELINE' not in globals():
    raise RuntimeError('Run the step-16 configuration cell first.')

_namespace = {'PIPELINE': PIPELINE}

_notebook = PIPELINE / '15_network_history_and_stationxml.ipynb'

for _cell in json.loads(_notebook.read_text())['cells']:
    if _cell['cell_type'] == 'code' and 'pipeline_definitions' in _cell.get('metadata', {}).get('tags', []):
        exec(compile(''.join(_cell['source']), str(_notebook), 'exec'), _namespace)

epochs = _namespace['build_channel_epochs']

nominal_response = _namespace['nominal_response']

load = _namespace['load']

inventory = _namespace['inventory']

from pinatubo_steps.config import pipeline_paths

ROOT=Path(pipeline_paths()['NETWORK_HISTORY_OUTPUT'])

class ReconstructionTests(unittest.TestCase):
 @classmethod
 def setUpClass(cls):
  import tempfile
  global ROOT
  cls._temp=tempfile.TemporaryDirectory(prefix='pinatubo-stationxml-')
  ROOT=Path(cls._temp.name)
  import pandas as pd
  headers=ROOT/'headers';headers.mkdir()
  pd.DataFrame([dict(file_month='07',station='PI2',channel='EHZ',date='1991-07-01',first='1991-07-01T00:00:00',last='1991-07-01T00:00:01',files=1)]).to_csv(headers/'waveform_header_presence.csv',index=False)
  _namespace['DEFAULT_HEADER_DATA']=headers
  for name,models in [('pinatubo_provisional.xml',False),('pinatubo_nominal_voltage_models.xml',True)]:
   inventory(epochs(),models=models).write(str(ROOT/name),format='STATIONXML')
 @classmethod
 def tearDownClass(cls):cls._temp.cleanup()

 def test_schema_and_unknown_electronics(self):
  for name in ['pinatubo_provisional.xml','pinatubo_nominal_voltage_models.xml']:
   valid,errors=validate_stationxml(str(ROOT/name));self.assertTrue(valid,errors)
  inv=read_inventory(str(ROOT/'pinatubo_provisional.xml'))
  self.assertEqual(len(inv[0]),13)
  self.assertTrue(all(c.response is None for s in inv[0] for c in s))
  self.assertEqual(len(inv.select(station='PI2',channel='ELZ')[0][0]),1)
  self.assertIsNone(inv.select(station='PI2',channel='ELZ')[0][0][0].start_date)
  self.assertEqual(len(inv.select(channel='WA?')[0].get_contents()['channels']),4)
 def test_calibration_response(self):
  for name in ['L4-C','L22-D']:
   a=nominal_response(name,18);b=nominal_response(name,24)
   # Evaluate stages, independently of the descriptive total sensitivity.
   av=a.get_evalresp_response_for_frequencies([5.],output='VEL')[0]
   bv=b.get_evalresp_response_for_frequencies([5.],output='VEL')[0]
   self.assertAlmostEqual(abs(bv/av),10**(-6/20),places=10)
   self.assertAlmostEqual(abs(av),a.instrument_sensitivity.value,places=8)
   self.assertEqual(a.instrument_sensitivity.output_units,'V')
 def test_epoch_integrity_and_low_gain(self):
  rows=epochs()
  for a in rows:
   if a['start'] and a['end']:self.assertLess(UTCDateTime(a['start']),UTCDateTime(a['end']))
   for b in rows:
    if a is b or (a['station'],a['channel'])!=(b['station'],b['channel']):continue
    if a['start'] and b['start'] and UTCDateTime(a['start'])<UTCDateTime(b['start']):
     self.assertIsNotNone(a['end']);self.assertLessEqual(UTCDateTime(a['end']),UTCDateTime(b['start']))
  self.assertTrue(all(r['attenuation_db'] is None for r in rows if r['sensor']=='unknown'))
 def test_coordinate_provenance(self):
  for s in load('stations.json'):
   self.assertTrue(14<s['latitude']<16);self.assertTrue(120<s['longitude']<121)
   self.assertNotIn(s['code'],['DFR','NCT']);self.assertTrue(s['coordinate_source'])

## 3. Run the controlled checks

Each test's result appears below. The report records the source notebook and reference locations so a pass can be tied to this configuration. A failed/error result raises after saving the summary; investigate it before relying on changed inventories.

In [ ]:
import hashlib
suite = unittest.defaultTestLoader.loadTestsFromTestCase(ReconstructionTests)
checks = unittest.TextTestRunner(verbosity=2).run(suite)
VALIDATION_OUTPUT.mkdir(parents=True, exist_ok=True)
summary = dict(created_utc=datetime.now(timezone.utc).isoformat(), tests_run=checks.testsRun,
               failures=len(checks.failures), errors=len(checks.errors), passed=checks.wasSuccessful(),
               failure_details=[dict(test=str(test), traceback=traceback) for test, traceback in checks.failures],
               error_details=[dict(test=str(test), traceback=traceback) for test, traceback in checks.errors],
               reference_root=P['NETWORK_HISTORY_INPUT'],
               step15_sha256=hashlib.sha256((PIPELINE / '15_network_history_and_stationxml.ipynb').read_bytes()).hexdigest())
(VALIDATION_OUTPUT / '16_controlled_checks.json').write_text(json.dumps(summary, indent=2) + '\n')
display(summary)
if not checks.wasSuccessful():
    raise AssertionError('Network reconstruction checks failed; see the saved validation report.')

## 4. Audit the actual saved inventories

This section requires step 15's saved files and never regenerates them. Missing files are reported as failures. Provisional/nominal documentary XML may share exact endpoints; positive-duration same-ID overlaps require review. Selection-safe copies should remove all such selection ambiguity. The audit reports units rather than claiming the voltage models provide a complete counts response.

The resulting JSON is an audit snapshot of these files, not an automatically refreshed certificate. Review undated channels and unresolved scientific questions in notebook 15 regardless of pass status.

In [ ]:
if RUN_SAVED_PRODUCT_AUDIT:
    audit_rows = {}
    failures = []
    names = ['pinatubo_provisional.xml', 'pinatubo_nominal_voltage_models.xml',
             'pinatubo_provisional_selection_safe.xml', 'pinatubo_nominal_voltage_models_selection_safe.xml']
    for name in names:
        path = NETWORK_OUTPUT / name
        if not path.is_file():
            audit_rows[name] = dict(status='missing', path=str(path))
            failures.append(name + ': missing')
            continue
        audit = _namespace['audit_inventory'](path)
        audit_rows[name] = dict(path=str(path), sha256=_namespace['sha256'](path), **audit)
        if not audit['schema_valid']:
            failures.append(name + ': invalid StationXML schema')
        unexpected = audit['overlaps'] if 'selection_safe' in name else [r for r in audit['overlaps'] if not r['boundary_only']]
        if unexpected:
            failures.append(name + ': unresolved epoch overlap')
    VALIDATION_OUTPUT.mkdir(parents=True, exist_ok=True)
    report = dict(created_utc=datetime.now(timezone.utc).isoformat(), inventories=audit_rows,
                  passed=not failures, failures=failures,
                  scope='Schema/epoch audit only; historical metadata and complete calibration not established.')
    (VALIDATION_OUTPUT / '16_saved_inventory_audit.json').write_text(json.dumps(report, indent=2) + '\n')
    display(report)
    if failures:
        raise AssertionError('Saved inventory audit requires attention: ' + '; '.join(failures))
else:
    print('Saved-product audit disabled; controlled tests alone do not validate the production XML files.')